# N147 · LRU与LFU缓存设计

**目标：** 使查询、更新和淘汰规则保持一致。

**先修：** N045, N050, N019, N014。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 哈希+双向链表；哨兵；最近使用；频次桶；最小频次；并列规则。

## 从问题到算法

哈希表负责按key定位，双向链表负责O(1)移动与淘汰，二者不能各保存一份独立状态。LRU维护访问先后；LFU先按频率选桶，再在桶内按LRU打破并列。更新已有key也算一次访问。容量为0时put不保存任何节点。

## 最小实现

**本章接口：** `LRUCache.get/put`、`LFUCache.get/put`

In [1]:
class _CacheNode:
    def __init__(self,key=None,value=None): self.key=key; self.value=value; self.frequency=1; self.prev=self.next=None

class _NodeList:
    def __init__(self):
        self.head=_CacheNode(); self.tail=_CacheNode(); self.head.next=self.tail; self.tail.prev=self.head; self.size=0
    def append(self,node):
        previous=self.tail.prev; previous.next=node; node.prev=previous; node.next=self.tail; self.tail.prev=node; self.size+=1
    def remove(self,node):
        node.prev.next=node.next; node.next.prev=node.prev; node.prev=node.next=None; self.size-=1
    def pop_left(self):
        if not self.size: raise IndexError('empty node list')
        node=self.head.next; self.remove(node); return node
class LRUCache:
    def __init__(self,capacity):
        if capacity<0: raise ValueError('nonnegative capacity required')
        self.capacity=capacity; self.nodes={}; self.order=_NodeList()
    def _touch(self,node): self.order.remove(node); self.order.append(node)
    def get(self,key):
        if key not in self.nodes: return -1
        node=self.nodes[key]; self._touch(node); return node.value
    def put(self,key,value):
        if key in self.nodes:
            node=self.nodes[key]; node.value=value; self._touch(node); return
        if not self.capacity: return
        if len(self.nodes)==self.capacity:
            victim=self.order.pop_left(); del self.nodes[victim.key]
        node=_CacheNode(key,value); self.nodes[key]=node; self.order.append(node)
class LFUCache:
    def __init__(self,capacity):
        if capacity<0: raise ValueError('nonnegative capacity required')
        self.capacity=capacity; self.nodes={}; self.buckets={}; self.minimum=0
    def _touch(self,node):
        old=node.frequency; bucket=self.buckets[old]; bucket.remove(node)
        if not bucket.size:
            del self.buckets[old]
            if self.minimum==old: self.minimum+=1
        node.frequency+=1; self.buckets.setdefault(node.frequency,_NodeList()).append(node)
    def get(self,key):
        if key not in self.nodes: return -1
        node=self.nodes[key]; self._touch(node); return node.value
    def put(self,key,value):
        if key in self.nodes:
            node=self.nodes[key]; node.value=value; self._touch(node); return
        if not self.capacity: return
        if len(self.nodes)==self.capacity:
            bucket=self.buckets[self.minimum]; victim=bucket.pop_left(); del self.nodes[victim.key]
            if not bucket.size: del self.buckets[self.minimum]
        node=_CacheNode(key,value); self.nodes[key]=node; self.buckets.setdefault(1,_NodeList()).append(node); self.minimum=1

## 正确性、前提与复杂度

哈希中的每个活节点恰出现于一条链表，移动只重连相邻指针，不创建重复节点。LRU访问移到末端，所以首端始终最久未访问。LFU访问仅从f桶移动到f+1桶；若最小桶因此变空，刚移动的节点保证新的最小频率为f+1。新节点频率1使minimum重置为1。

**代价：** 哈希操作平均O(1)，所有链表操作O(1)，所以每次get/put平均O(1)，空间O(capacity)。这是顺序调用接口，未提供并发访问锁；测试同时检查返回值和映射/双向链表结构。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

cache=LRUCache(2); trace=[]
for op in [('put',1,1),('put',2,2),('get',1),('put',3,3),('get',2)]:
    result=getattr(cache,op[0])(*op[1:]); keys=[]; node=cache.order.head.next
    while node is not cache.order.tail: keys.append(node.key); node=node.next
    trace.append((op,result,keys))
show_table(['操作','返回','LRU→MRU顺序'],trace)

操作,返回,LRU→MRU顺序
"('put', 1, 1)",None,[1]
"('put', 2, 2)",None,"[1, 2]"
"('get', 1)",1,"[2, 1]"
"('put', 3, 3)",None,"[1, 3]"
"('get', 2)",-1,"[1, 3]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from collections import OrderedDict
from random import Random
rng=Random(147)
def inspect(bucket):
    nodes=[]; previous=bucket.head; node=previous.next
    while node is not bucket.tail:
        assert node.prev is previous and previous.next is node
        assert node not in nodes; nodes.append(node); previous,node=node,node.next
    assert bucket.tail.prev is previous and bucket.size==len(nodes)
    backward=[]; node=bucket.tail.prev
    while node is not bucket.head: backward.append(node); node=node.prev
    assert backward==nodes[::-1]
    return nodes
for capacity in range(5):
    lru=LRUCache(capacity); slow=OrderedDict(); lfu=LFUCache(capacity); reference={}; clock=0
    for _ in range(500):
        key=rng.randrange(8); clock+=1
        if rng.randrange(2):
            value=rng.randrange(100); lru.put(key,value); lfu.put(key,value)
            if capacity:
                if key in slow: del slow[key]
                elif len(slow)==capacity: slow.popitem(last=False)
                slow[key]=value
                if key in reference: old,f,t=reference[key]; reference[key]=(value,f+1,clock)
                else:
                    if len(reference)==capacity: del reference[min(reference,key=lambda k:(reference[k][1],reference[k][2]))]
                    reference[key]=(value,1,clock)
        else:
            expected=slow.get(key,-1)
            if key in slow: slow.move_to_end(key)
            assert lru.get(key)==expected
            expected=reference[key][0] if key in reference else -1
            if key in reference:
                value,f,t=reference[key]; reference[key]=(value,f+1,clock)
            assert lfu.get(key)==expected
        assert [node.key for node in inspect(lru.order)]==list(slow)
        assert set(lru.nodes.values())==set(inspect(lru.order))
        seen=[]
        for frequency,bucket in lfu.buckets.items():
            nodes=inspect(bucket); seen.extend(nodes); assert all(node.frequency==frequency for node in nodes)
            expected=sorted((k for k in reference if reference[k][1]==frequency),key=lambda k:reference[k][2])
            assert [node.key for node in nodes]==expected
        assert set(seen)==set(lfu.nodes.values()) and len(seen)==len(lfu.nodes)==len(reference)
        if reference: assert lfu.minimum==min(v[1] for v in reference.values())
print('N147: 所有本章断言通过')

N147: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 更新已存在键时正确刷新优先级。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 用慢速时间戳模型作为参照。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 146. LRU Cache（canonical）
- 460. LFU Cache（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。